# TMotorAPI — 제어 템플릿 (VER2)

CubeMars AK 모터 제어 템플릿. **500 Hz 제어 루프 + 별도 스레드 상태출력 + 정밀 슬립**.
`TEMPLATE.ipynb`의 골격을 그대로 두고, 각 상황·제어 모드 사용법을 주석으로 설명한다.

## 사용 순서
1. `MotorConfig` 로 모터/CAN/**드라이버 버전** 설정
2. `motor.enable()` → `motor.zero_position()` (영점)
3. 매 루프: `motor.update()` (송수신) → `motor.set_*()` (원하는 제어 **하나** 선택)

## 드라이버 버전 (v2 / v3)
- **v2 보드**: `driverVersion` 생략(기본 `"v2"`)
- **v3 보드**(AK driver v3.x): `driverVersion="v3"`
  - v3 지원 모터: `AK10-9`(실측) · `AK60-6` · `AK60-39` · `AK70-9` · `AK80-8` · `AK80-9` · `AKE60-8` · `AKE80-8` · `AKE90-8` · `AKH70-16` · `AKH70-48`
  - 목록 밖 모터로 v3 → `KeyError` (그 모터는 v3 매뉴얼에 없음)

## 제어 모드 (한 루프에 **하나만** 활성화)
| 모드 | 메서드 | 언제 |
|---|---|---|
| 위치 | `set_position(pos, kp, kd, feedTor)` | 특정 각도로 이동/유지 |
| 속도 | `set_velocity(vel, kd)` | 일정 속도 회전 |
| 토크 | `set_torque(tor)` | 힘/토크 직접 (오픈) |
| 풀스테이트 | `set_fullState(pos, vel, kp, kd, feedTor)` | 위치+속도+피드포워드 한 번에 |
| 소프트리밋 위치 | `set_soft_limit(...)` + `set_position_smooth(t)` | 범위 밖 못 나가게 |

> ⚠️ **안전**: PSU 전류제한 낮게(1~3A), 모터 고정. Ctrl+C / 커널 interrupt 로 안전 종료(finally에서 토크0 + disable).
> v3는 **영점을 먼저** 잡아야 위치제어가 안전하다(범위 밖 절대명령 시 폭주 방지 게이트 작동).

In [ ]:
from TMotorAPI import Motor, MotorConfig
import time
import threading
import math
import ctypes

# ── Windows 타이머 해상도 1ms로 설정 ──────────────
#    (리눅스면 이 두 줄 삭제. Windows 전용)
winmm = ctypes.WinDLL('winmm')
winmm.timeBeginPeriod(1)

# ── 모터 설정 ───────────────────────────
cfg = MotorConfig(
    motorType="AK70-10",   # 모델. v3면 지원목록에 있는 모델이어야 함(AK10-9 등)
    motorId=1,             # CAN ID (0~127). CubeMarsTool에서 설정한 값과 일치
    canBackend="gs_usb",   # Windows: "gs_usb" / Linux: "socketcan"
    canChannel=0,          # gs_usb: 0 (정수) / socketcan: "can0" (문자열)
    bitrate=1_000_000,     # 1 Mbps
    maxTemperature=80,     # MOSFET 안전 온도 상한(°C). 초과 시 update()가 raise
    autoInit=False,        # Windows는 False (ip link 불필요)
    # driverVersion="v3",  # ← AK driver v3.x 보드면 주석 해제 (기본 "v2")
    # defaultKp=15.0,      # set_position 에서 kp 생략 시 기본값
    # defaultKd=1.5,       # set_position/set_velocity 에서 kd 생략 시 기본값
)
motor = Motor(config=cfg)
motor.enable()
motor.zero_position()   # 현재 위치를 0으로 재정의(모터 안 움직임).
                        # v3는 서보 Set Origin 사용 + 안전게이트(영점 실패 시 홀드 스킵)
                        # 수동으로 축을 돌려보려면 루프에서 set_torque(0) 사용

# ── (선택) 소프트리밋 ───────────────────────
#    set_position_smooth() 를 쓸 거면 루프 진입 전 1회 설정.
# motor.set_soft_limit(-1.57, 1.57)   # -90° ~ +90°, 한계 근처 댐핑 자동 증가

# ── 컨트롤 루프 설정 ─────────────────────
TARGET_HZ    = 500
PERIOD       = 1.0 / TARGET_HZ
BUSYWAIT_THR = 0.001

RAD_S_TO_RPM = 60.0 / (2.0 * math.pi)   # rad/s → rpm 변환 계수

# ── 공유 상태 ───────────────────────────
stop_event = threading.Event()
state_lock = threading.Lock()
shared = {"pos": 0.0, "vel": 0.0, "tor": 0.0, "hz": 0.0, "misses": 0, "err": 0}

# ── 상태 출력 (별도 스레드, 5Hz) ───────────────
def status_printer():
    while not stop_event.is_set():
        with state_lock:
            s = shared.copy()
        rpm = s['vel'] * RAD_S_TO_RPM
        err = s['err']
        err_str = "OK" if err == 0 else f"FAULT({err})"   # 0 = 정상, 그 외 = 폴트코드
        print(
            f"Pos: {math.degrees(s['pos']):8.2f}°  "
            f"Vel: {rpm:8.2f} rpm  "
            f"Tor: {s['tor']:6.3f} Nm  "
            f"Loop: {s['hz']:5.1f} Hz  "
            f"Miss: {s['misses']:3d}  "
            f"Err: {err_str}"
        )
        time.sleep(0.2)

threading.Thread(target=status_printer, daemon=True).start()

# ── 정밀 슬립 ──────────────────────────
def precise_sleep_until(target: float):
    remaining = target - time.perf_counter()
    if remaining > BUSYWAIT_THR:
        time.sleep(remaining - BUSYWAIT_THR)
    while time.perf_counter() < target:
        pass

# ── 메인 제어 루프 ───────────────────────
try:
    print(f"[INFO] 모터 테스트 ({TARGET_HZ} Hz)")
    last_time = time.perf_counter()
    next_time = last_time
    misses = 0

    while not stop_event.is_set():
        now = time.perf_counter()
        dt = now - last_time
        last_time = now

        # update(): 직전 명령 송신 + 최신 상태 수신. 매 루프 필수.
        motor.update()

        ##################  제어 모드 — 아래 중 하나만 활성화  ##################

        # ── 1) 위치 제어 (Position + Feedforward Torque) ──
        #    pos_cmd(rad)로 이동/유지. kp=강성(클수록 딱 붙음), kd=댐핑, feedTor=피드포워드 토크.
        #    ⚠ v3는 영점 먼저! 범위(±12.56 rad) 밖 절대명령은 게이트가 막음.
        # pos_cmd = math.radians(90)             # 목표 각도(deg→rad)
        # motor.set_position(pos_cmd, kp=15.0, kd=1.5, feedTor=0.0)

        # ── 2) 속도 제어 (Velocity) ──
        #    vel_cmd(rad/s)로 회전. kd=속도 게인.
        # vel_cmd = math.radians(360)            # 360 deg/s
        # motor.set_velocity(vel_cmd, kd=0.5)

        # ── 3) 토크 제어 (Torque) ──  [기본 활성]
        #    tau_cmd(Nm) 그대로 출력. 위치/속도 피드백 없는 오픈.
        tau_cmd = 3.00
        motor.set_torque(tau_cmd)

        # ── 4) 풀스테이트 (Full-state MIT) ──
        #    tau = kp*(pos-p) + kd*(vel-v) + feedTor 를 한 번에. kp=0이면 속도댐핑+피드포워드.
        # motor.set_fullState(targetPos=math.radians(45), targetVel=0.0,
        #                     kp=10.0, kd=1.0, feedTor=0.0)

        # ── 5) 소프트리밋 위치 (범위 밖으로 못 나가게) ──
        #    루프 전에 motor.set_soft_limit(-1.57, 1.57) 1회 호출 필요.
        # motor.set_position_smooth(math.radians(90))   # 리밋 근처 자동 댐핑

        ##################################################################

        with state_lock:
            shared.update({
                "pos": motor.position,
                "vel": motor.velocity,
                "tor": motor.torque,
                "hz": 1.0 / dt if dt > 0 else 0.0,
                "misses": misses,
                "err": motor.error,     # 0 = OK, 그 외 = 드라이버 폴트코드
            })

        next_time += PERIOD
        slack = next_time - time.perf_counter()
        if slack > 0:
            precise_sleep_until(next_time)
        else:
            misses += 1

except KeyboardInterrupt:
    print("\n[INFO] 종료 중...")

finally:
    # 안전 종료: 토크 0 + disable (v3는 부드러운 감속 후 정식 disable)
    stop_event.set()
    time.sleep(0.3)
    motor.set_torque(0.0)
    motor.update()
    motor.disable()
    winmm.timeEndPeriod(1)
    print("[INFO] 완료")